# 🐸 Wednesday Frog – Flow Matching Training (Kaggle / Colab)

Two stages: **1. pretrain** on frog photos + memes (learns what frogs look like), **2. fine-tune** on the memes only.

**Kaggle:** Settings → Accelerator = *GPU T4 x2* or *P100*, Internet = *On*. Under *Add Input* add two datasets of your own:
- `wednesday-frogs` ← `data/wednesday-frogs.zip`
- `frog-photos` ← `data/pretrain/frog-photos.zip`

(Both zips may also live together in **one** dataset – the notebook looks for the folders `wednesday-frogs/128` and `frog-photos/128`, no matter what the dataset is called.)

**Colab:** Runtime → GPU. Put both zips into Google Drive (`MyDrive/`).

On a timeout simply run everything again – `train.py` automatically continues at `last.pt` (as long as the output folder survives, e.g. on Drive).

In [ ]:
import os
REPO = "https://github.com/alexanderulbrich/wednesday.git"
# Always work with absolute paths – that way this cell may run any number of times (first time: clone, afterwards: update).
WORK = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
%cd {WORK}
!git clone -q $REPO wednesday 2>/dev/null || git -C wednesday pull -q
%cd {WORK}/wednesday
!pip install -q -e .
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, os

def find_dataset(name):
    """The folder name/128 with the images – wherever Kaggle/Colab mounts it.

    The folder directly above 128/ must be called exactly `name`. That works with two separate datasets
    (/kaggle/input/frog-photos/128) just as well as with both zips in one shared dataset
    (/kaggle/input/whatever/frog-photos/128).
    """
    hits = sorted(glob.glob(f"{ROOT}/**/{name}/128", recursive=True))
    if not hits:
        raise FileNotFoundError(f"'{name}/128' not found under {ROOT} – dataset added as input? "
                                f"Check the contents with: !find {ROOT} -maxdepth 5 -type d")
    if len(hits) > 1:
        print(f"Warning, several matches for {name}, taking the first: {hits}")
    return hits[0]

if os.path.exists("/kaggle"):
    ROOT, OUT = "/kaggle/input", "/kaggle/working"
else:  # Colab
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, OUT = "/content/data", "/content/drive/MyDrive/wednesday-runs"   # runs on Drive -> survive timeouts
    for z in ("wednesday-frogs", "frog-photos"):
        !mkdir -p {ROOT}/{z} && unzip -qo /content/drive/MyDrive/{z}.zip -d {ROOT}/{z}

MEMES, PHOTOS = find_dataset("wednesday-frogs"), find_dataset("frog-photos")
RES = 64   # training happens at 64 px, the source are the sharper 128 px images
for d in (MEMES, PHOTOS):
    print(d, len(os.listdir(d)), "images")

## Speed test (~6 min)
The same small network, once with image preparation on the CPU, once on the GPU, plus the big network on the GPU. The interesting lines are `step … | xx it/s | data xx %`: if the data share is large with the CPU and the GPU is clearly faster, data loading is the bottleneck.

In [ ]:
!python train.py --data {PHOTOS},{MEMES} --res {RES} --base 32 --steps 500 --loader cpu --out {OUT}/speed_cpu --val-every 100000 --sample-every 100000 --save-every 100000 --log-every 100
!python train.py --data {PHOTOS},{MEMES} --res {RES} --base 32 --steps 500 --loader gpu --out {OUT}/speed_gpu --val-every 100000 --sample-every 100000 --save-every 100000 --log-every 100
!python train.py --data {PHOTOS},{MEMES} --res {RES} --base 64 --steps 300 --loader gpu --out {OUT}/speed_gpu64 --val-every 100000 --sample-every 100000 --save-every 100000 --log-every 100

## Quick prototype (~30 min)
A smaller network (`--base 32`, ~4.5M instead of 18M parameters) and a time budget: `--minutes 25` measures the speed after ~120 steps and sets the number of steps so that training finishes after 25 minutes. Its own folder `proto/`, so nothing collides with the big run. Afterwards create frogs right below.

In [ ]:
!python train.py --data {PHOTOS},{MEMES} --res {RES} --base 32 --minutes 25 --steps 100000 --lr 3e-4 --warmup 500 --batch 64 --out {OUT}/proto --sample-every 1000 --val-every 1000 --save-every 2000 --workers 2

In [ ]:
# Create frogs from the prototype + show the loss curves
!python generate.py --ckpt {OUT}/proto/frog.pt --n 16 --grid --size 128 --no-open --out {OUT}/proto/generated
from IPython.display import Image, display
display(Image(f"{OUT}/proto/loss.png"))
display(Image(sorted(glob.glob(f"{OUT}/proto/generated/*.png"))[-1]))

## Stage 1: pretrain (frog photos + memes)

In [ ]:
!python train.py --data {PHOTOS},{MEMES} --res {RES} --out {OUT}/pre --steps 40000 --batch 64 --sample-every 2000 --save-every 5000 --workers 2

## Stage 2: fine-tune (memes only)
Shorter and with a smaller learning rate. Check in the sample grids from when on the images only copy training images – then rather take an earlier state.

In [ ]:
!python train.py --init {OUT}/pre/last.pt --data {MEMES} --res {RES} --out {OUT}/ft --steps 10000 --lr 1e-4 --warmup 200 --batch 64 --sample-every 1000 --save-every 2000 --workers 2

## Look at the progress
While a training is running the notebook is blocked – then open `loss.png` and `samples/` on the right under *Output*.

In [ ]:
from IPython.display import Image, display
RUN = "ft"   # or "pre"
# Loss curves (left: training, right: training vs. validation images; explained in wednesday/monitor.py)
if os.path.exists(f"{OUT}/{RUN}/loss.png"):
    display(Image(f"{OUT}/{RUN}/loss.png"))
for s in sorted(glob.glob(f"{OUT}/{RUN}/samples/*.png"))[-3:]:
    print(s); display(Image(s))

At the end download `frog.pt` (EMA weights only, small) from the output and run `python generate.py --ckpt frog.pt` on the Mac.

In [ ]:
# Export an intermediate state as frog.pt (in case a training is not finished yet)
import torch
RUN = "ft"   # or "pre"
ck = torch.load(f"{OUT}/{RUN}/last.pt", map_location="cpu", weights_only=False)
torch.save({"ema": ck["ema"], "model_config": ck["model_config"], "step": ck["step"]}, f"{OUT}/{RUN}/frog.pt")
print("step", ck["step"], "->", f"{OUT}/{RUN}/frog.pt")